<a href="https://colab.research.google.com/github/LikaOkudzhava/BrainyScan/blob/xception_model/Alzheimer_model_inceptionV3_Version3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Alzheimer's Dataset Classification using InceptionV3
This notebook trains an InceptionV3 model on MRI images to classify Alzheimer's stages.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [8]:
import os

import zipfile

from google.colab import drive

gdrive = '/content/drive'

gdisk = os.path.join(gdrive, 'MyDrive')

filename = 'smallpreprocessed'

archive_path = os.path.join(gdisk, filename)

with zipfile.ZipFile('/content/drive/MyDrive/AI/smallpreprocessed.zip', 'r') as zip_ref:

 zip_ref.extractall(f'{filename}')

In [4]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt

# Parameters
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
DATASET_DIR = "/content/smallpreprocessed/data/SmallPreprocessed"


In [3]:
# Load datasets
train_ds = image_dataset_from_directory(f"{DATASET_DIR}/train", image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical')
val_ds = image_dataset_from_directory(f"{DATASET_DIR}/val", image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical')
test_ds = image_dataset_from_directory(f"{DATASET_DIR}/test", image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical')

# Prefetch
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)


NameError: name 'image_dataset_from_directory' is not defined

In [2]:
# Build InceptionV3 model
base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = tf.keras.applications.inception_v3.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(4, activation='softmax')(x)

model = models.Model(inputs, outputs)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


NameError: name 'InceptionV3' is not defined

In [1]:
# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)

# Training
history = model.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=[early_stop, reduce_lr])


NameError: name 'EarlyStopping' is not defined

In [ ]:
# Evaluation
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test Accuracy: {test_acc:.2%}")


In [ ]:
# Optional fine-tuning
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

history_fine = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=[early_stop, reduce_lr])


In [ ]:
# Plotting training history
def plot_history(hist):
    plt.plot(hist.history['accuracy'], label='Train Acc')
    plt.plot(hist.history['val_accuracy'], label='Val Acc')
    plt.plot(hist.history['loss'], label='Train Loss')
    plt.plot(hist.history['val_loss'], label='Val Loss')
    plt.legend()
    plt.title('Training & Validation Accuracy and Loss')
    plt.xlabel('Epochs')
    plt.show()

plot_history(history)
